#LEVEL 1 — BASE AGGREGATED FACTS

##IMPORT CONFIG

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import lag

##Sales Daily (CORE TABLE)

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_sales_daily",
    comment="Daily sales using fully validated FK joins",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_sales_daily():

    tx = dlt.read("maven_market_uc.silver.silver_transactions")

    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True) \
              .select("product_id")

    cust = dlt.read("maven_market_uc.silver.silver_dim_customers_scd") \
              .filter(col("__IS_CURRENT") == True) \
              .select("customer_id")

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True) \
               .select("store_id")

    valid_tx = (
        tx
        .join(prod, "product_id", "inner")
        .join(cust, "customer_id", "inner")
        .join(store, "store_id", "inner")
    )

    return (
        valid_tx
        .groupBy(
            "transaction_date",
            "store_id",
            "product_id"
        )
        .agg(
            sum("quantity").alias("total_units"),
            count("*").alias("total_transactions")
        )
        # ✅ ADD HERE (very important)
        .withColumn("processed_time", current_timestamp())
    )

##Returns Daily

In [0]:
from pyspark.sql.functions import *

@dlt.table(
    name="maven_market_uc.gold.gold_returns_daily",
    comment="Daily aggregated returns fact table with FK validation",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_returns_daily():

    # Source (clean Silver)
    returns = dlt.read("maven_market_uc.silver.silver_returns")

    # Dimension tables (ONLY keys for FK validation)
    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True) \
              .select("product_id")

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True) \
               .select("store_id")

    # ✅ FK validation (ensures only valid records)
    valid_returns = (
        returns
        .join(prod, "product_id", "inner")
        .join(store, "store_id", "inner")
    )

    # ✅ Aggregation (Gold layer logic)
    return (
        valid_returns
        .groupBy(
            "return_date",
            "store_id",
            "product_id"
        )
        .agg(
            sum("quantity").alias("returned_units"),
            count("*").alias("return_transactions")
        )
        # ✅ Metadata for observability
        .withColumn("processed_time", current_timestamp())
    )

##Inventory Snapshot

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_inventory_snapshot",
    comment="Latest inventory snapshot per store and product with FK validation",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_inventory_snapshot():

    # Source (Silver cleaned streaming data)
    inv = dlt.read("maven_market_uc.silver.silver_inventory")

    # Dimension tables (ONLY keys for FK validation)
    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True) \
              .select("product_id")

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True) \
               .select("store_id")

    # ✅ FK validation (ensures valid product & store only)
    valid_inv = (
        inv
        .join(prod, "product_id", "inner")
        .join(store, "store_id", "inner")
    )

    # ✅ Snapshot logic (latest inventory per store-product)
    return (
        valid_inv
        .groupBy("store_id", "product_id")
        .agg(
            max("quantity_remaining").alias("current_stock"),
            sum("quantity_added").alias("total_added")
        )
        # ✅ metadata for observability
        .withColumn("processed_time", current_timestamp())
    )

#LEVEL 2 — ENRICHED TABLE

##Sales Enriched

In [0]:
from pyspark.sql.functions import *

@dlt.table(
    name="maven_market_uc.gold.gold_sales_enriched",
    comment="Enriched sales fact with dimension attributes for BI consumption",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_sales_enriched():

    sales = dlt.read("maven_market_uc.gold.gold_sales_daily")

    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True)

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True)

    region = dlt.read("maven_market_uc.silver.silver_regions")

    enriched = (
        sales
        .join(prod, "product_id", "left")
        .join(store, "store_id", "left")
        .join(region, "region_id", "left")
    )

    return enriched.withColumn("processed_time", current_timestamp())

this wll create a wide table.

correct table to impplement if column names turn out same:

In [0]:
from pyspark.sql.functions import *

@dlt.table(
    name="maven_market_uc.gold.gold_sales_enriched",
    comment="Enriched sales fact with dimension attributes for BI consumption",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_sales_enriched():

    # Base fact
    sales = dlt.read("maven_market_uc.gold.gold_sales_daily")

    # Select ONLY required columns from dimensions (important)
    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True) \
              .select(
                  "product_id",
                  "product_name",
                  "product_category"
              )

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True) \
               .select(
                   "store_id",
                   "store_name",
                   "region_id"
               )

    region = dlt.read("maven_market_uc.silver.silver_regions") \
                .select(
                    "region_id",
                    "sales_region",
                    "sales_district"
                )

    # ✅ Proper joins
    enriched = (
        sales
        .join(prod, "product_id", "left")
        .join(store, "store_id", "left")
        .join(region, "region_id", "left")
    )

    return (
        enriched
        .withColumn("processed_time", current_timestamp())
    )

#THIS IS CORRECT CODE

In [0]:
##gold_returns_enriched

In [0]:


@dlt.table(
    name="maven_market_uc.gold.gold_returns_enriched",
    comment="Enriched returns fact with product and store details",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_returns_enriched():

    returns = dlt.read("maven_market_uc.gold.gold_returns_daily")

    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True)

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True)

    region = dlt.read("maven_market_uc.silver.silver_regions")

    enriched = (
        returns
        .join(prod, "product_id", "left")
        .join(store, "store_id", "left")
        .join(region, "region_id", "left")
    )

    return enriched.withColumn("processed_time", current_timestamp())

##gold_inventory_enriched

In [0]:

@dlt.table(
    name="maven_market_uc.gold.gold_inventory_enriched",
    comment="Enriched inventory snapshot with product and store details",
    table_properties={
        "delta.enableLiquidClustering": "true"
    }
)
def gold_inventory_enriched():

    inv = dlt.read("maven_market_uc.gold.gold_inventory_snapshot")

    prod = dlt.read("maven_market_uc.silver.silver_dim_products_scd") \
              .filter(col("__IS_CURRENT") == True)

    store = dlt.read("maven_market_uc.silver.silver_dim_stores_scd") \
               .filter(col("__IS_CURRENT") == True)

    region = dlt.read("maven_market_uc.silver.silver_regions")

    enriched = (
        inv
        .join(prod, "product_id", "left")
        .join(store, "store_id", "left")
        .join(region, "region_id", "left")
    )

    return enriched.withColumn("processed_time", current_timestamp())

#NEED TO CHANGE THESE TO REMOVE WIDE TABLE ISSUE

#HERE WE START KPI

###1. EXECUTIVE SUMMARY DASHBOARD
- Purpose
  -High-level view for leadership
- Uses
  -gold_sales_enriched
- KPIs
  -	Total Units Sold
  -	Total Transactions
  -	Daily/Monthly Trend
  -	Top Performing Region
  -	Top Product
  -	Growth % (day/week/month)
  -	Active Stores
  -	Sales Distribution
  - Sales Distribution = visualization (pie/bar chart)


##EXECUTIVE SUMMARY TABLE

In [0]:
#from pyspark.sql.functions import *

@dlt.table(
    name="maven_market_uc.gold.gold_kpi_executive_summary",  # ✅ DO NOT CHANGE unless your catalog/schema differs
    comment="Executive summary KPIs for leadership dashboard"
)
def gold_kpi_executive_summary():

    df = dlt.read("maven_market_uc.gold.gold_sales_enriched")  # ✅ MUST MATCH your table name

    return (
        df.groupBy("transaction_date")
        .agg(
            sum("total_units").alias("total_units_sold"),
            sum("total_transactions").alias("total_transactions"),
            countDistinct("store_id").alias("active_stores"),
            countDistinct("product_id").alias("unique_products")
        )
        .withColumn("processed_time", current_timestamp())
    )

##TOP PRODUCT TABLE

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_top_products",
    comment="Top performing products by sales"
)
def gold_kpi_top_products():

    df = dlt.read("maven_market_uc.gold.gold_sales_enriched")

    return (
        df.groupBy("product_id")
        .agg(
            sum("total_units").alias("units_sold"),
            sum("total_transactions").alias("transactions")
        )
    )

Finds best-selling products ✔
Used in dashboard ranking ✔

##TOP STORE TABLE

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_top_stores",
    comment="Top performing stores"
)
def gold_kpi_top_stores():

    df = dlt.read("maven_market_uc.gold.gold_sales_enriched")

    return (
        df.groupBy("store_id")
        .agg(
            sum("total_units").alias("units_sold"),
            sum("total_transactions").alias("transactions")
        )
    )

##GROWTH % TABLE

In [0]:
from pyspark.sql.window import Window

@dlt.table(
    name="maven_market_uc.gold.gold_kpi_sales_growth",
    comment="Day-over-day sales growth"
)
def gold_kpi_sales_growth():

    df = dlt.read("maven_market_uc.gold.gold_kpi_executive_summary")

    window_spec = Window.orderBy("transaction_date")

    return (
        df
        .withColumn("prev_units", lag("total_units_sold").over(window_spec))
        .withColumn(
            "growth_pct",
            when(col("prev_units").isNull(), None)
            .otherwise(
                (col("total_units_sold") - col("prev_units")) / col("prev_units") * 100
            )
        )
    )

##Monthly Trend

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_monthly_sales"
)
def gold_kpi_monthly_sales():

    df = dlt.read("maven_market_uc.gold.gold_sales_enriched")

    return (
        df
        .withColumn("month", date_format(col("transaction_date"), "yyyy-MM"))
        .groupBy("month")
        .agg(
            sum("total_units").alias("monthly_units"),
            sum("total_transactions").alias("monthly_transactions")
        )
    )

##5. Top Performing Region

👉 NOT DONE YET

Reason:

We did not confirm region column name
✔ Action

After pipeline runs:

display(dlt.read("maven_market_uc.gold.gold_sales_enriched"))

👉 Then tell me region column → I’ll give exact code


#Problem:

We don’t know if your column is:

sales_region
region_name
region
✅ SAFE WAY (DO THIS AFTER PIPELINE RUNS)

After pipeline runs:

display(dlt.read("maven_market_uc.gold.gold_sales_enriched"))

👉 Then tell me column name
👉 I’ll give exact code

#📊 2. PRODUCT PERFORMANCE DASHBOARD
🎯 Purpose
Understand which products drive business
📌 Uses
gold_sales_enriched, gold_returns_enriched
📊 KPIs
  -	Units Sold per Product
  -	Top/Bottom Products
  -	Return Rate per Product
  -	Product Category Performance
  -	Profitability Proxy (price vs volume)
  -	Product Demand Trend
  -	Product Contribution %
  •	Fast-moving vs slow-moving items
  ________________________________________


#1. UNITS SOLD PER PRODUCT (BASE TABLE)

In [0]:
from pyspark.sql.functions import *

@dlt.table(
    name="maven_market_uc.gold.gold_kpi_product_sales",
    comment="Units sold and transactions per product"
)
def gold_kpi_product_sales():

    df = dlt.read("maven_market_uc.gold.gold_sales_enriched")

    return (
        df.groupBy("product_id")
        .agg(
            sum("total_units").alias("units_sold"),
            sum("total_transactions").alias("transactions")
        )
    )

##7. PRODUCT CATEGORY PERFORMANCE

👉 We are SKIPPING for now because:

We don’t know your column name yet

After pipeline runs, we will add:

product_category / brand / type
⚠️ 8. PROFITABILITY (PRICE VS COST)

👉 Also SKIPPED for now

Because we don’t know if you have:

product_cost / product_price

##TOP / BOTTOM PRODUCTS

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_top_bottom_products",
    comment="Top and bottom performing products"
)
def gold_kpi_top_bottom_products():

    df = dlt.read("maven_market_uc.gold.gold_kpi_product_sales")

    return df.orderBy(col("units_sold").desc())

##3. RETURNS PER PRODUCT

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_product_returns",
    comment="Returns per product"
)
def gold_kpi_product_returns():

    df = dlt.read("maven_market_uc.gold.gold_returns_enriched")

    return (
        df.groupBy("product_id")
        .agg(
            sum("returned_units").alias("total_returns")
        )
    )

##4. RETURN RATE PER PRODUCT (VERY IMPORTANT KPI)

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_product_return_rate",
    comment="Return rate per product"
)
def gold_kpi_product_return_rate():

    sales = dlt.read("maven_market_uc.gold.gold_kpi_product_sales")
    returns = dlt.read("maven_market_uc.gold.gold_kpi_product_returns")

    return (
        sales
        .join(returns, "product_id", "left")
        .withColumn("total_returns", coalesce(col("total_returns"), lit(0)))
        .withColumn(
            "return_rate_pct",
            when(col("units_sold") == 0, 0)
            .otherwise((col("total_returns") / col("units_sold")) * 100)
        )
    )

##5. PRODUCT DEMAND TREND (TIME BASED)

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_product_trend",
    comment="Product demand over time"
)
def gold_kpi_product_trend():

    df = dlt.read("maven_market_uc.gold.gold_sales_enriched")

    return (
        df.groupBy("transaction_date", "product_id")
        .agg(
            sum("total_units").alias("units_sold")
        )
    )

##6. PRODUCT CONTRIBUTION %

In [0]:
@dlt.table(
    name="maven_market_uc.gold.gold_kpi_product_contribution",
    comment="Product contribution percentage"
)
def gold_kpi_product_contribution():

    df = dlt.read("maven_market_uc.gold.gold_kpi_product_sales")

    total_units = df.agg(sum("units_sold").alias("total")).collect()[0]["total"]

    return (
        df.withColumn(
            "contribution_pct",
            (col("units_sold") / lit(total_units)) * 100
        )
    )

#3. CUSTOMER ANALYTICS DASHBOARD
- Purpose
Customer behavior & segmentation
- Uses
gold_sales_enriched
📊 KPIs
  -	Total Customers
  -	Orders per Customer
  -	Avg Purchase Frequency
  -	Customer Segmentation (income, education)
  -	Gender-wise Sales
  -	Repeat vs New Customers
  -	Customer Lifetime Value (approx)
  -	Region-wise customer distribution
________________________________________


##Enrichment....move this to gold

In [0]:
@dlt.table
def transactions_enriched():

    tx = dlt.read("silver_transactions")
    cust = dlt.read("silver_customers")
    prod = dlt.read("silver_products")
    store = dlt.read("silver_stores")
    region = dlt.read("silver_regions")

    return tx.join(prod, "product_id") \
             .join(cust, "customer_id") \
             .join(store, "store_id") \
             .join(region, "region_id")